# BirdCLEF 2026 — BirdNET embedding + Head B MLP inference\n\nOffline notebook using only Head B (MLP). Upload bundle as dataset and set MODEL_DIR.

In [ ]:
%pip install -q birdnet librosa scikit-learn tensorflow

In [ ]:
import os\nfrom pathlib import Path\n\nimport librosa\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nimport birdnet\n\nCOMP_DIR = '/kaggle/input/birdclef-2026'\nMODEL_DIR = '/kaggle/input/REPLACE_WITH_YOUR_DATASET_FOLDER/submission_augmented_birdnet'\n\nmeta = np.load(os.path.join(MODEL_DIR, 'metadata.npz'), allow_pickle=True)\nspecies_cols = list(meta['species_cols'])\nEMB_DIM = int(meta['emb_dim'])\nSR_LOAD = int(meta['sr_load'])\nBIRDNET_SR = int(meta['birdnet_sr'])\nCLIP_LOAD_SEC = float(meta['clip_load_sec'])\nBIRDNET_CHUNK_SEC = float(meta['birdnet_chunk_sec'])\n\n# Load only Head B\nmlp_b = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'head_b_mlp.keras'), compile=False)\n\nbird_model = birdnet.load('acoustic', '2.4', 'tf')\nbird_sr = int(bird_model.get_sample_rate())\nBIRDNET_BATCH_SIZE = 32\nbird_sess = bird_model.encode_session(batch_size=BIRDNET_BATCH_SIZE, prefetch_ratio=2, n_workers=2, n_producers=1)\nbird_sess.__enter__()\n\ndef audio_to_birdnet_chunk(y, sr):\n    y = np.asarray(y, dtype=np.float32).reshape(-1)\n    if sr != BIRDNET_SR:\n        y = librosa.resample(y, orig_sr=sr, target_sr=BIRDNET_SR)\n    max_keep = int(CLIP_LOAD_SEC * BIRDNET_SR)\n    if len(y) > max_keep:\n        y = y[:max_keep]\n    need = int(BIRDNET_CHUNK_SEC * BIRDNET_SR)\n    if len(y) >= need:\n        s0 = max(0, (len(y) - need) // 2)\n        y = y[s0:s0+need]\n    else:\n        y = np.pad(y, (0, need - len(y)))\n    return y.astype(np.float32)\n\ndef extract_feat(y, sr):\n    chunk = audio_to_birdnet_chunk(y, sr).astype(np.float32)\n    res = bird_sess.run_arrays([(chunk, bird_sr)])\n    feat = np.asarray(res.embeddings[0, 0, :], dtype=np.float32).reshape(-1)\n    if feat.shape[0] != EMB_DIM:\n        raise RuntimeError(f'Unexpected BirdNET embedding dim {feat.shape}; expected {EMB_DIM}')\n    return feat\n\ndef predict_heads(X):\n    # Only use Head B\n    pb = mlp_b.predict(X, batch_size=512, verbose=0)\n    return pb\n

In [ ]:
sample_sub = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))\ntest_dir = Path(COMP_DIR) / 'test_soundscapes'\n\npred_rows = []\nfor rid in sample_sub['row_id']:\n    stem, end_s = rid.rsplit('_', 1)\n    end_sec = int(end_s)\n    start_sec = max(0, end_sec - int(CLIP_LOAD_SEC))\n    fpath = test_dir / f'{stem}.ogg'\n    y, _ = librosa.load(str(fpath), sr=SR_LOAD, mono=True, offset=start_sec, duration=CLIP_LOAD_SEC)\n    n = int(CLIP_LOAD_SEC * SR_LOAD)\n    if len(y) > n:\n        y = y[:n]\n    elif len(y) < n:\n        y = np.pad(y, (0, n - len(y)))\n    feat = extract_feat(y, SR_LOAD)[None, :]\n    pred = predict_heads(feat)[0]\n    row = {'row_id': rid}\n    for c, p in zip(species_cols, pred):\n        row[c] = float(np.clip(p, 0.0, 1.0))\n    pred_rows.append(row)\n\nsub = pd.DataFrame(pred_rows)\nsub = sample_sub[['row_id']].merge(sub, on='row_id', how='left')\nsub[species_cols] = sub[species_cols].fillna(0.0).clip(0.0, 1.0)\nsub.to_csv('/kaggle/working/submission.csv', index=False)\nprint('Saved /kaggle/working/submission.csv', sub.shape)